The data sources include: NVIDIA's historical stock price from yfinance for API and S&P 500 company information from Wikipedia for web scraping.

## Documentation
### Data Sources

API data
- ticker: NVDA
- source: Yahoo Finance (yfinance)
- data: historical stock price data
- param: start = `2024-01-01`, end = `2026-08-17`, auto_adjust = `False`

web scraping data:
- source: wikipedia: list of S&P 500 companies
- table: S&P 500 constituent companies
- URL: https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
- parameter: table id = `constituents`

### Validation
- API data was checked for required cols, missing values, and DataFrame shape
- scraped data were checked for required cols, missing values, shape, and text data types
- raw datasets saved as CSV files in `data/raw/`

# Assumptions
I assume the data from Yahoo Finance is accurate and available for the selected dates. I also assume the S&P 500 table on Wikipedia keep the same format. If Yahoo Finance or Wiki changes their data or website structure, the code may not work properly.

In [3]:
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup

In [5]:
ticker = "NVDA"
start_date = "2024-01-01"
end_date = "2026-08-17"
print(ticker, start_date, end_date)

NVDA 2024-01-01 2026-08-17


In [6]:
nvda_data = yf.download(ticker, start = start_date, end = end_date, auto_adjust = False)
nvda_data.head()

[*********************100%***********************]  1 of 1 completed


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,,
2024-01-02,48.082531,48.167999,49.294998,47.595001,49.243999,411254000
2024-01-03,47.484596,47.569000,48.183998,47.320000,47.485001,320896000
2024-01-04,47.912842,47.998001,48.500000,47.507999,47.766998,306535000
2024-01-05,49.009884,49.097000,49.547001,48.306000,48.462002,415039000
2024-01-08,52.160282,52.252998,52.275002,49.479000,49.512001,642510000


In [15]:
print("sjape", nvda_data.shape)
print("\nData types:")
print(nvda_data.dtypes)

print("\nMissing values:")
print(nvda_data.isna().sum())

sjape (657, 7)

Data types:
Price
Date         datetime64[s]
Adj Close          float64
Close              float64
High               float64
Low                float64
Open               float64
Volume               int64
dtype: object

Missing values:
Price
Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64


In [11]:
required_col = ["Open", "High", "Low", "Close", "Volume"]
actual_col = nvda_data.columns.get_level_values(0)
missing_col = [col for col in required_col if col not in actual_col]
print("required col", required_col, "missing col", missing_col)

required col ['Open', 'High', 'Low', 'Close', 'Volume'] missing col []


In [12]:
nvda_data.columns = nvda_data.columns.get_level_values(0)
nvda_data = nvda_data.reset_index()
nvda_data.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume
0,2024-01-02,48.082531,48.167999,49.294998,47.595001,49.243999,411254000
1,2024-01-03,47.484596,47.569000,48.183998,47.320000,47.485001,320896000
2,2024-01-04,47.912842,47.998001,48.500000,47.507999,47.766998,306535000
3,2024-01-05,49.009884,49.097000,49.547001,48.306000,48.462002,415039000
4,2024-01-08,52.160282,52.252998,52.275002,49.479000,49.512001,642510000


In [13]:
print(nvda_data.dtypes)

Price
Date         datetime64[s]
Adj Close          float64
Close              float64
High               float64
Low                float64
Open               float64
Volume               int64
dtype: object


In [14]:
from pathlib import Path
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
print(raw_dir)

data/raw


In [18]:
from datetime import datetime
time_stamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
file_path = raw_dir / f"nvda_{time_stamp}.csv"
nvda_data.to_csv(file_path, index=False)
print("Saved", file_path)

Saved data/raw/nvda_2026-08-19_07-22-19.csv


In [33]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/151.0.0.0 Safari/537.36"
}
response = requests.get(url, headers=headers)
print("Status code:", response.status_code)

Status code: 200


In [34]:
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table")
print("Number of tables:", len(tables))

Number of tables: 2


In [35]:
table = soup.find('table', id='constituents')
rows = []

for tr in table.find_all('tr'):
    cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
    if cells:
        rows.append(cells)

header, *data = rows
df_scrape = pd.DataFrame(data, columns=header)

df_scrape.head()

,Symbol,Security,GICSSector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,0001467373,1989


In [36]:
print(df_scrape.shape)
print(df_scrape.columns)
print(df_scrape.isna().sum())

(503, 8)
Index(['Symbol', 'Security', 'GICSSector', 'GICS Sub-Industry',
       'Headquarters Location', 'Date added', 'CIK', 'Founded'],
      dtype='str')
Symbol                   0
Security                 0
GICSSector               0
GICS Sub-Industry        0
Headquarters Location    0
Date added               0
CIK                      0
Founded                  0
dtype: int64


In [39]:
df_scrape = df_scrape.rename(columns={"GICSSector": "GICS Sector"})

In [40]:
required_cols = ["Symbol", "Security", "GICS Sector"]
missing_cols = [col for col in required_cols if col not in df_scrape.columns]
print("missing req col:", missing_cols)

missing req col: []


In [41]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
file_path = f"data/raw/sp500_{timestamp}.csv"
df_scrape.to_csv(file_path, index=False)
print("save", file_path)

save data/raw/sp500_2026-08-20_22-01-44.csv


In [43]:
print(df_scrape.dtypes)

Symbol                   str
Security                 str
GICS Sector              str
GICS Sub-Industry        str
Headquarters Location    str
Date added               str
CIK                      str
Founded                  str
dtype: object


In [45]:
text_cols = ["Symbol", "Security", "GICS Sector"]
for col in text_cols:
    print(col, df_scrape[col].dtype)

Symbol str
Security str
GICS Sector str
